# Build 15-Minute BTC Dataset

This notebook:
1. Loads all 1-minute trading_data.csv files from `historical data just in case/BTCUSDT`
2. Standardizes, cleans, and concatenates into one 1-minute dataset
3. Converts to 15-minute OHLCV
4. Saves outputs to `data/raw/` and `data/processed/`

**Assumptions:**
- Timestamps are in UTC (no timezone conversion applied)
- Source: Binance-style spot trading_data.csv (Open time, Open, High, Low, Close, Volume, Number of trades, Taker buy base asset volume)

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# Resolve project root (works from notebook or script)
PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "historical data just in case").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
SOURCE_DIR = PROJECT_ROOT / "historical data just in case" / "BTCUSDT"
RAW_OUT = PROJECT_ROOT / "data" / "raw" / "btc_1m_combined.csv"
PROCESSED_OUT = PROJECT_ROOT / "data" / "processed" / "btc_15m.csv"

## 1. Load and Inspect Source Files

In [ ]:
csv_files = sorted(SOURCE_DIR.glob("**/trading_data.csv"))
print(f"Found {len(csv_files)} trading_data.csv files")
for f in csv_files[:5]:
    print(f"  {f.relative_to(SOURCE_DIR)}")
print("  ...")
for f in csv_files[-3:]:
    print(f"  {f.relative_to(SOURCE_DIR)}")

In [ ]:
# Schema inspection: first file
sample = pd.read_csv(csv_files[0], nrows=3)
print("Schema (columns):")
print(sample.columns.tolist())
print("\nSample:")
print(sample)

## 2. Preprocessing Pipeline

In [ ]:
COLUMN_MAP = {
    "Open time": "open_time",
    "Open": "open",
    "High": "high",
    "Low": "low",
    "Close": "close",
    "Volume": "volume",
    "Number of trades": "num_trades",
    "Taker buy base asset volume": "taker_buy_volume",
}

def load_and_clean(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path)
    df = df.rename(columns=COLUMN_MAP)
    df["open_time"] = pd.to_datetime(df["open_time"], errors="coerce")
    df = df.dropna(subset=["open_time"])
    return df

dfs = []
loaded_files = []
for p in csv_files:
    try:
        df = load_and_clean(p)
        dfs.append(df)
        loaded_files.append(str(p.relative_to(PROJECT_ROOT)))
    except Exception as e:
        print(f"Error loading {p}: {e}")

combined = pd.concat(dfs, ignore_index=True)
print(f"Loaded {len(loaded_files)} files")
print(f"Combined shape before dedup: {combined.shape}")

In [ ]:
# Sort chronologically
combined = combined.sort_values("open_time").reset_index(drop=True)

# Remove duplicates (keep first)
n_before = len(combined)
combined = combined.drop_duplicates(subset=["open_time"], keep="first")
n_duplicates = n_before - len(combined)
print(f"Duplicates removed: {n_duplicates}")

# Missing values report
missing = combined.isnull().sum()
print("\nMissing values per column:")
print(missing[missing > 0] if missing.any() else "None")

# Drop rows with any NaN in OHLCV (conservative)
cols_ohlcv = ["open", "high", "low", "close", "volume"]
combined = combined.dropna(subset=cols_ohlcv)
print(f"\nFinal 1m row count: {len(combined)}")
print(f"Date range 1m: {combined['open_time'].min()} to {combined['open_time'].max()}")

## 3. Save Combined 1-Minute Dataset

In [ ]:
RAW_OUT.parent.mkdir(parents=True, exist_ok=True)
combined.to_csv(RAW_OUT, index=False)
print(f"Saved: {RAW_OUT}")
print(f"Rows: {len(combined)}")

## 4. Resample to 15-Minute OHLCV

In [ ]:
combined = combined.set_index("open_time")

agg_dict = {
    "open": "first",
    "high": "max",
    "low": "min",
    "close": "last",
    "volume": "sum",
}
# Additional columns: conservative handling
if "num_trades" in combined.columns:
    agg_dict["num_trades"] = "sum"
if "taker_buy_volume" in combined.columns:
    agg_dict["taker_buy_volume"] = "sum"

df_15m = combined.resample("15min").agg(agg_dict).dropna(how="all")
df_15m = df_15m.dropna(subset=cols_ohlcv)
df_15m = df_15m.reset_index()

print(f"15m row count: {len(df_15m)}")
print(f"Date range 15m: {df_15m['open_time'].min()} to {df_15m['open_time'].max()}")
print("\n15m columns:", df_15m.columns.tolist())

In [ ]:
PROCESSED_OUT.parent.mkdir(parents=True, exist_ok=True)
df_15m.to_csv(PROCESSED_OUT, index=False)
print(f"Saved: {PROCESSED_OUT}")

## 5. Preprocessing Report

In [ ]:
report = {
    "files_loaded": len(loaded_files),
    "file_list_sample": loaded_files[:3] + ["..."] + loaded_files[-2:],
    "schema": list(COLUMN_MAP.values()),
    "inconsistencies_fixed": [
        "Column names standardized to snake_case",
        "Timestamps parsed with pd.to_datetime",
        "Rows sorted chronologically",
    ],
    "duplicates_removed": n_duplicates,
    "missing_values": missing[missing > 0].to_dict() if missing.any() else {},
    "date_range_1m": (str(combined.index.min()), str(combined.index.max())),
    "date_range_15m": (str(df_15m["open_time"].min()), str(df_15m["open_time"].max())),
    "row_count_1m": len(combined),
    "row_count_15m": len(df_15m),
    "assumptions": [
        "Timestamps assumed UTC (no conversion)",
        "trading_data.csv used (spot), not futures_data.csv",
        "num_trades and taker_buy_volume summed in 15m aggregation",
    ],
}

for k, v in report.items():
    print(f"{k}: {v}")